# Bangladesh: 0233_sen5_no2

Static adaptation only; Earth Engine/data/asset execution not verified.

Original scientific assumptions and date windows retained; use the national workflow for QA improvements.

Review scale, missing data, units, and any training/calibration requirements before interpreting results.

May contain tutorial-specific assets or imported training features; inspect dependencies before execution.

In [ ]:
# Bangladesh adaptation: 0233_sen5_no2
# Original author: Amirhossein Ahrari; see original attribution below and originals/.
from bdgee.core import initialize, boundaries
from bdgee.notebook import export_outputs, upload_local
initialize()  # Run `earthengine authenticate` first; project is configured in bdgee.
BD_COUNTRY, BD_DISTRICTS = boundaries()
BD_ROI = BD_COUNTRY.geometry()
BD_BBOX = [88.0, 20.5, 92.7, 26.7]  # search only, not analysis boundary
roi = geometry = border = iran = tehran = province = loc = point = BD_ROI
# Run from the repository root; all local outputs are isolated by source.
from pathlib import Path
import os
_REPO_ROOT = Path.cwd()
_OUTPUT_DIR = Path(os.environ.get('BDGEE_OUTPUT_DIR', _REPO_ROOT / 'outputs' / 'air_quality' / '0233_sen5_no2'))
_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(_OUTPUT_DIR)


In [ ]:
"""00022_sen5_no2.ipynb

Tutorial Code by Amirhossein Ahrari
YouTube: https://www.youtube.com/@amirhosseinahrarigee
Tutorial Video: Google Earth Engine Tutorial-209: Nitrogen Dioxide (No2) Analysis using Sentinel-5P
This code is part of a tutorial series on Earth Engine programming techniques
presented on the Amirhossein Ahrari YouTube channel. You are free to use and modify
this code for academic and non-academic purposes. Don't forget to subscribe to
the Amirhossein Ahrari channel and follow the videos to support the instructor


"""
import ee
None
None
'# **Study Area**'
import geemap
map = geemap.Map()
map
point = BD_ROI
point
tehran = BD_ROI
tehran
map.addLayer(tehran, {}, 'Tehran')
'# **Sentienl-5 NO2 Image Collection**'
no2 = ee.ImageCollection('COPERNICUS/S5P/NRTI/L3_NO2').filterBounds(tehran).filterDate('2020', '2026').select(['NO2_column_number_density'], ['no2'])
no2.size()
no2.limit(100)
'# **to Xarray**'
from xee import helpers
from shapely.geometry import shape
import xarray as xr
grid_1km = helpers.fit_geometry(geometry=shape(tehran.getInfo()), grid_crs='EPSG:4326', grid_scale=(0.01, -0.01))
ds = xr.open_dataset(no2, engine='ee', **grid_1km, chunks='auto')
ds
from dask.diagnostics import ProgressBar
with ProgressBar():
    ds = ds.compute()
'# **Clip**'
import geopandas as gpd
import rioxarray
tehran_gdf = geemap.ee_to_gdf(ee.FeatureCollection(tehran))
tehran_gdf.plot()
ds_clip = ds.rio.clip(tehran_gdf.geometry, tehran_gdf.crs)
ds_clip
'# **Preview**'
ds_mean = ds_clip.mean(dim='time')
ds_mean.no2.plot(x='x', y='y', cmap='Spectral_r', robust=True)
'# **Multi - Temporal Analysis**'
ds_month = ds_clip.groupby('time.month').mean('time')
ds_month.no2.plot(x='x', y='y', robust=True, col='month', col_wrap=6, cmap='Spectral_r')
ds_annual = ds_clip.resample(time='YE').mean('time')
ds_annual.no2.plot(x='x', y='y', robust=True, col='time', cmap='Spectral_r', col_wrap=3)
ds_anomaly = ds_annual - ds_annual.mean('time')
ds_anomaly.no2.plot(x='x', y='y', robust=True, col='time', col_wrap=3, cmap='Spectral_r')
ds_annual_diff = ds_annual.diff('time')
ds_annual_diff.no2.plot.contourf(x='x', y='y', robust=True, col='time', col_wrap=3, cmap='Spectral_r', levels=20)
'# **Point Based Monitoring**'
ds_daily = ds.resample(time='D').mean('time')
df = ds_daily.sel(x=51.36883410111137, y=35.720296441621386, method='nearest').to_dataframe().no2
df.to_csv('no2_point.csv')
import matplotlib.pyplot as plt
thresh = df.quantile(0.95)
extreme = df[df >= thresh]
extreme
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(df.index, df)
ax.scatter(extreme.index, extreme, color='red')


In [ ]:

# Review these task plans; task creation does not start exports.
os.chdir(_REPO_ROOT)
bd_tasks = export_outputs(globals(), 'air_quality', '0233_sen5_no2', scale=1000)
# Set an appropriate native scale above before creating plans.
# for task in bd_tasks: task.start()
# upload_local(_OUTPUT_DIR, 'air_quality')  # explicit upload; requires Drive OAuth scope
